# 06: Range Trees

*Authors: Felix Espey, Kevin Buchin*

This notebook serves as supplementary learning material for the course **Geometric Algorithms**.
It showcases and explains implementations of algorithms presented in the corresponding lecture, and elaborates on some practical considerations concerning their use.
Furthermore, it offers interactive visualisations and animations.

## Table of Contents

1. Introduction
2. Range Searching in one Dimension
3. Range Searching in two Dimensions (Kd-Trees)
4. Range Trees
5. References

## 1. Introduction



In [1]:
from ctypes.wintypes import tagPOINT

from modules.data_structures.binary_trees import BST, EST, TreeTracker, TransitionType, TransitionEvent
from modules.geometry import IntComparator, Point
import random
from modules.visualisation import VisualisationTool, BinaryTreeInstance, BinaryTreeMode


bst = BST[int](IntComparator(), True)
est = EST[int](IntComparator(), True)

keys = [3,10,19,23,30,27,59,62,70,80,100,105]
random.shuffle(keys)
for k in keys:
    est.insert(k)

lo = est.level_order()

for l in lo:
    s = ""
    for o in l:
        s += str("None" if o is None else o) + " | "
    print(s)

#print(est.level_order())
#print(est._find_split_node((12,35)).key)
#print([node.key for node in est.range_query((12,35))])
#print(est.size)
print(est.in_order(lambda n : n._level))

bti = BinaryTreeInstance()
vis = VisualisationTool(400,400,bti)
vis.display()


62 | 
19 | 100 | 
10 | 27 | 70 | 105 | 
3 | 19 | 23 | 30 | 62 | 80 | 100 | 105 | 
3 | 10 | None | None | 23 | 27 | 30 | 59 | None | None | 70 | 80 | None | None | None | None | 
[None, 0, None, 1, None, 0, None, 2, None, 0, None, 3, None, 0, None, 1, None, 0, None, 2, None, 0, None, 1, None, 0, None, 4, None, 0, None, 2, None, 0, None, 1, None, 0, None, 3, None, 0, None, 1, None, 0, None]


In [2]:
from modules import Point
from modules.geometry.core import Comparator, ComparisonResult

t = TreeTracker()
lower_bound = 60
upper_bound = 100
est = bti._instance

#points = [10, 20, 40,80,160,240,380]
#for point in points:
    #est.insert(point)

print(est.in_range(40, 110, t, lambda n : n._key))
#print([event.__str__() for event in t.events])


print(t.get_routine_events("first_in_range"))


[56, 82, 89]
[[<modules.data_structures.binary_trees.base.node.NodeVisitedEvent object at 0x000002667EDC0050>, <modules.data_structures.binary_trees.base.node.NodeVisitedEvent object at 0x000002667ED760D0>, <modules.data_structures.binary_trees.base.node.NodeVisitedEvent object at 0x000002667ED76210>, <modules.data_structures.binary_trees.base.node.ResultAddedEvent object at 0x000002667EDC01A0>, <modules.data_structures.binary_trees.base.node.NodeVisitedEvent object at 0x000002667ED888A0>, <modules.data_structures.binary_trees.base.node.NodeVisitedEvent object at 0x000002667ED889D0>]]


In [9]:
from modules import Node, IntComparator, TreeTracker
from typing import Optional

int_comparator = IntComparator()

def less_or_equal(node : Node[int, None], upper_bound : int) -> list[Node[int, None]]:
    cr = int_comparator.compare(upper_bound, node.key)
    if cr is ComparisonResult.MATCH or cr is ComparisonResult.AFTER:
        #less than search term
        if not node.is_leaf():
            return node.left.leaves(TreeTracker(), lambda n : n) + less_or_equal(node.right, upper_bound)
        else:
            return [node]
    else:
        #more than search term
        if not node.is_leaf():
            return less_or_equal(node.left, upper_bound)
        else:
            return []

def greater_or_equal(node : Node[int, None], lower_bound : int) -> list[Node[int, None]]:
    cr = int_comparator.compare(lower_bound, node.key)
    if cr is ComparisonResult.BEFORE or cr is ComparisonResult.MATCH:
        #less than search term
        if not node.is_leaf():
            return greater_or_equal(node.left, lower_bound) + node.right.leaves(TreeTracker(), lambda n : n)
        else:
            return [node]
    else:
        #more than search term
        if not node.is_leaf():
            return greater_or_equal(node.right, lower_bound)
        else:
            return []

def find_splitting_node(node : Node[int, None], lower_bound : int, upper_bound : int) -> Optional[Node[int, None]]:
    cr_left = int_comparator.compare(lower_bound, node.key)
    cr_right = int_comparator.compare(upper_bound, node.key)
    if cr_right is ComparisonResult.BEFORE:
        #range fully left of node
        if node.left is None:
            return None
        return find_splitting_node(node.left, lower_bound, upper_bound)
    elif cr_left is ComparisonResult.AFTER:
        #range fully right of node
        if node.right is None:
            return None
        return find_splitting_node(node.right, lower_bound, upper_bound)
    else:
        #node in range
        return node

def range_search(node : Node[int, None], lower_bound: int, upper_bound : int) -> list[Node[int, None]]:
        if node is None:
            return []
        if lower_bound >= upper_bound:
            return []
        splitting_node = find_splitting_node(node, lower_bound, upper_bound)
        if splitting_node is None:
            return []
        print(splitting_node.key)
        if splitting_node.is_leaf():
            return [splitting_node]
        else:
            result = []
            if splitting_node.left is not None:
                result+= greater_or_equal(splitting_node.left, lower_bound)
            if splitting_node.right is not None:
                result += less_or_equal(splitting_node.right, upper_bound)
            return result


print([str(node.key) for node in range_search(est._root, 10, 340)])


206
259
352
335
335
['56', '82', '89', '135', '158', '206', '236', '259', '335']


## 2. Range Searching in one Dimension

TODO

## 3. Range Searching in two Dimensions (Kd-Trees)
TODO

## 4. Range Trees
TODO